In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
from astropy.io import fits
import os
import numpy as np
import jax.numpy as jnp

import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
tfb = tfp.bijectors

import matplotlib.pyplot as plt

In [ ]:
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.profiles.mass.epl import EPL
from gigalens.jax.profiles.mass.shear import Shear
from gigalens.jax.profiles.mass.nfw import NFW_ELLIPSE, NFW_ELLIPSE_EINSTEIN
from gigalens.jax.profiles.mass.nfw_ellipse_slope import NFW_ELLIPSE_SLOPE
from gigalens.jax.profiles.mass.piemd import DPIE

from gigalens.jax.profiles.light.sersic import SersicEllipse
from gigalens.jax.profiles.light.shapelets import Shapelets

from gigalens.jax.cosmo import wCDM_Cosmo

from gigalens.jax.scene_prob_model import Dataset, ImageData, ProbModel
from gigalens.simulator import SimulatorConfig

In [ ]:
from gigalens.jax.utils.grouped_priors import DiskEllipticity
from gigalens.jax.experimental.adaptive_supersample import AdaptiveImageData, plot_factor_map

In [ ]:
from translate_old_params import build
from real_datasets import real_simulators_for_model
from gigalens.jax.utils.noise import add_noise
import jax

model, spec = build("sersic-simulated.json", zero_negative_amplitudes=False)
rsims = real_simulators_for_model(model, spec, "real_cutouts", delta_pix=0.2, supersample=16)
sims = [s.simulator for s in rsims]


In [ ]:
model.to_params({})

In [ ]:
p = model.to_params({})
src11light = p['planes']['source11']['light']['source11']
# srclight['center_x'] = 2.8828425
# srclight['center_y'] = 1.1054275
src11light['R_sersic'] = 0.16
src11light['n_sersic'] = 2.
src11light['Ie'] = 58813.41015625/130 # peak PSF convolved brightness should be ~12

# # src1neglight = p['planes'][1]['light'][0]
# # # src1neglight['Ie'] = 0.0
# # src1light['R_sersic'] = 0.1
# # src1light['n_sersic'] = 0.8

src1light = p['planes']['source1_2']['light']['source1']
src1light['R_sersic'] = 0.08
src1light['n_sersic'] = 0.8

src1light['Ie'] /= 1.5

# src2light = p['planes'][1]['light'][1]
# src2light['n_sersic']=0.5
# src2light['Ie']/=2
# src2light['R_sersic']=0.4
# # src1light['n_sersic'] = 1.
# # for pl in p['planes'].values():

# src4light= p['planes'][3]['light'][0]
# src4light['n_sersic'] = 0.5
# src4light['Ie'] *= 1.5


# src5light= p['planes'][3]['light'][1]
# src5light['n_sersic'] = 0.5
# src5light['Ie'] *= 1.5

# src3neglight = p['planes']['source3']['light']['source3_core']
# src3neglight['n_sersic']=0.5
# src3neglight['Ie']/=3

src9light= p['planes']['source9']['light']['source9']
src9light['n_sersic'] = 2.0
src9light['Ie'] *= 2.

In [ ]:
from gigalens_research.plotting import plot_scene
from gigalens.jax.scene_simulator import SceneSimulator

In [ ]:

cfg_k = SimulatorConfig(delta_pix=0.2, num_pix=300, supersample=1, kernel=None, likelihood_precision="float64")
sims_nopsf = [SceneSimulator(model, cfg_k, sees=model.planes[i].light) for i in range(1, len(model.planes))]
figs = plot_scene(model, sims_nopsf, p, with_curves=False)#, fov_arcsec={2: 5.0, 7: 10.0}, center={1: (8.0, 3.5)})
plt.show()

In [ ]:
figs = plot_scene(model, sims[:1], p, with_curves=False)#, fov_arcsec={2: 5.0, 7: 10.0}, center={1: (8.0, 3.5)})
plt.show()

In [ ]:
import ersatz_carousel_prior_new_api
import importlib
importlib.reload(ersatz_carousel_prior_new_api)
from ersatz_carousel_prior_new_api import model as model_free
from ersatz_carousel_prior_new_api import source1_plane, source3_plane, source45_plane, source9_plane, source7_plane, source6_plane, source1213_plane, source8_plane, source11_plane, cluster_plane, cosmo

src_planes = [source1_plane, source3_plane, source45_plane, source9_plane,
        source7_plane, source6_plane, source1213_plane, source8_plane, source11_plane]

conservative_snr_levels = (
    (20.0, 8.0),
    (10.0, 4.0),
    (7.0, 2.0),
    (-np.inf, 1.0),
)

snr_levels = (
    (15.0, 8.0),
    (8.0, 4.0),
    (6.0, 2.0),
    (2.0, 1.0),
    (1.0, 0.5),
    (-np.inf, 0.25),
)


import copy
datasets = []
key = jax.random.PRNGKey(0)
for i, entry, subkey in zip(range(len(sims)), rsims, jax.random.split(key, len(sims))):
    image = entry.simulator.simulate(p) #* simulate with supersample=8
    noisy = add_noise(subkey, image, entry.exp_time, entry.background_rms)

    light = src_planes[i].light
    cfg = copy.deepcopy(entry.simulator.sim_config)

    cfg.supersample=1
    if i in [4,5,6,8]:
        datasets.append(AdaptiveImageData(noisy, cfg, exp_time=entry.exp_time, background_rms=entry.background_rms, sees=light, snr_levels=conservative_snr_levels))
    else:
        datasets.append(AdaptiveImageData(noisy, cfg, exp_time=entry.exp_time, background_rms=entry.background_rms, sees=light, snr_levels=snr_levels))



In [ ]:
filter_i = list(range(len(datasets)))
filtered_datasets = [datasets[i] for i in filter_i]

model_filtered = LensModel(
    [cluster_plane, *[src_planes[i] for i in filter_i]],
    cosmo=cosmo, unconstrain='gaussian',
)
for d in filtered_datasets:
    plot_factor_map(d.adaptive_grid)

prob_model = ProbModel(model_filtered, filtered_datasets, mode="lstsq")

In [ ]:
from gigalens.jax.analysis import diagnose_undersampling

In [ ]:
import copy

def remove_key(pytree, key_to_remove):
    """
    Recursively copy a nested-dict pytree, dropping any dict entry
    whose key equals `key_to_remove` at any depth.
    """
    if isinstance(pytree, dict):
        return {
            k: remove_key(v, key_to_remove)
            for k, v in pytree.items()
            if k != key_to_remove
        }
    elif isinstance(pytree, (list, tuple)):
        cls = type(pytree)
        return cls(remove_key(v, key_to_remove) for v in pytree)
    else:
        # leaf node (e.g. a jax/np Array, float, etc.) -> copy as-is
        return copy.deepcopy(pytree)


# Example usage:
# new_tree = remove_key(original_tree)

rep = diagnose_undersampling(prob_model, remove_key(p, 'Ie'), reference_supersample=16, dataset_idxes=[0])

In [ ]:
for r in rep:
    r.plot()

In [ ]:
fig, axs = plt.subplots(1, len(datasets))
fig.set_size_inches(25, 3)
for ax, d in zip(axs, datasets):
    im = ax.imshow(d.image/d.error_map)
    fig.colorbar(im, ax=ax)

plt.show()

In [ ]:
# from gigalens.jax.analysis.preflight import sampling_preflight

# res = sampling_preflight(prob_model, p, run_gradient = False,
#                        run_stiffness = False,
#                        run_gates= False,)

In [ ]:
# print(res.summary())

In [ ]:
res.plot()

In [ ]:
from gigalens_research.inference_utils import *

z_truth = prob_model.unconstrained(p)
if jnp.any(~jnp.isfinite(z_truth)):
    raise ValueError("NAN IN Z")

ctx_in= InferenceContext.from_prob_model(prob_model)
pipeline = Pipeline(ctx_in, seed=0)

# def make_diag_qz(z_best):
#     return tfd.MultivariateNormalDiag(
#         loc=jnp.asarray(z_best),
#         scale_diag=jnp.full(z_best.shape[-1], 1e-3),
#     )

# pipeline.add(BridgeStage(
#     name="diag_qz_from_map",
#     version="v1",
#     requires=("z_best",),
#     produces=("qz",),
#     fn=make_diag_qz,
# ))

pipeline.add(SVIStage(
    num_steps=2000,
    n_vi=300,               
    init_scales=1e-3,       
    pbar_interval=5,
))

pipeline.add(MCLMCStage(
    n_chains=8,
    num_burnin_steps=20000,
    num_results=20000,
    desired_energy_variance=5e-4,
    seed=10,
    progress_bar=True,
    debug=True,
    regularize_mass_matrix=True,
))
# pipeline.add(MAMSStage(
#     n_chains=8,
#     num_burnin_steps=2000,
#     num_results=2000,
#     seed=10,
#     progress_bar=True,
#     debug=True,
#     regularize_mass_matrix=True,
# ))

In [ ]:
results_dir = os.path.join(
    os.path.expanduser("~"), "GIGALens-Code", "results",
    "ersatz_carousel", "startfit",
)
artifacts = pipeline.run(out_dir=results_dir,seed_artifacts={"z_best": z_truth}, resume=True)


In [ ]:
# import gigalens_research
# importlib.reload(gigalens_research.plotting)
from gigalens_research.plotting import PosteriorReport, PipelineReport
report = PosteriorReport(pipeline.posterior(), truth_x=p)
report.corner(plot_params=['cosmo/Om0', 'cosmo/w0', 'cosmo/wa'])#kind="cosmology")
report.convergence_panel(n_worst=5)

# report.source_comparison_panel()
# report.full_report()
plt.show()

In [ ]:
jnp.sum(pipeline.posterior().rhat > 1.01)

In [ ]:
report.image_panel()
report.source_panel()
plt.show()

In [ ]:
report.corner(kind="mass")
plt.show()

In [ ]:
import gigalens_research
importlib.reload(gigalens_research.plotting.diagnostics)
from gigalens_research.plotting import PosteriorReport, PipelineReport
pipeline_report = PipelineReport(pipeline)
fig = pipeline_report.diagnostics("mclmc", chain=3)
fig.show()

In [ ]:
pipeline_report.loss_histories()
plt.show()